In [1]:
import psycopg2
import os
import io
from dotenv import load_dotenv
import pandas as pd
import sqlalchemy
import matplotlib.pyplot as plt

In [2]:
load_dotenv(dotenv_path="../../.env")

host=os.getenv("host")
dbname=os.getenv("dbname")
user=os.getenv("user")
password=os.getenv("password")
port=os.getenv("port")

engine = sqlalchemy.create_engine(f'postgresql://{user}:{password}@{host}:{port}/{dbname}')


1) Qual é o total de consultas realizadas por especialidade?

In [3]:
query_1 = """
        SELECT especialidade,
        count(*) AS tot_consulta
        FROM case_2.consultas_silver
        GROUP BY especialidade
        ORDER BY tot_consulta DESC
        """
Q1 = pd.read_sql(query_1, engine)

Q1

,especialidade,tot_consulta
0,ortopedia,81
1,cardiologia,65
2,dermatologia,46
3,ginecologia,36
4,pediatria,34
5,neurologia,32
6,oftalmologia,31
7,psiquiatria,29


2) Qual convênio gerou mais receita total para a clínica?

In [4]:
query_2 = """
        SELECT convenio,
        sum(valor_consulta) AS receita_convenio
        FROM case_2.consultas_silver
        GROUP BY convenio
        ORDER BY receita_convenio DESC
        """
Q2 = pd.read_sql(query_2, engine)

Q2

,convenio,receita_convenio
0,sulamérica,22100.0
1,bradesco saúde,21040.0
2,unimed,19940.0
3,particular,19800.0
4,amil,15490.0


3) Qual médico tem a melhor avaliação média entre os que têm ao menos 10 avaliações válidas?

In [5]:
query_3 = """
    WITH t1 AS (
        SELECT nome_medico,
               avaliacao_paciente::numeric
        FROM case_2.consultas_silver
        WHERE avaliacao_paciente NOT IN ('invalido', 'nao avaliado')
    )
    SELECT nome_medico, 
           round(avg(avaliacao_paciente),2) AS media_avaliacao
    FROM t1
    GROUP BY nome_medico
    HAVING count(*) >= 10
    ORDER BY media_avaliacao DESC
    LIMIT 1
"""
Q3 = pd.read_sql(query_3, engine)

Q3

,nome_medico,media_avaliacao
0,renata souza,3.27


4) Qual a taxa de cancelamento por especialidade mês a mês? Existe alguma especialidade com tendência crescente de cancelamentos?

In [73]:
query_4 = """
    WITH t1 AS (SELECT 
    TO_CHAR(data_consulta, 'YYYY-MM') AS ano_mes,
    especialidade,
    count(status) AS qtd_cancelado
    FROM case_2.consultas_silver
    WHERE status = 'cancelada'
    GROUP BY ano_mes, especialidade),

    meses AS (SELECT 
    DISTINCT TO_CHAR(data_consulta, 'YYYY-MM') AS ano_mes,
    especialidade
    FROM case_2.consultas_silver),

    t3 AS (SELECT
    meses.ano_mes,
    meses.especialidade,
    COALESCE(t1.qtd_cancelado,0) AS qtd_cancelado
    FROM meses
    LEFT JOIN t1
    ON t1.ano_mes=meses.ano_mes
    AND t1.especialidade=meses.especialidade
    ORDER BY especialidade, ano_mes),

    t3_fix AS (SELECT *,
    SUM(COALESCE(qtd_cancelado,0)) OVER (PARTITION BY especialidade ORDER BY ano_mes) AS qtd_cancelado_acum
    FROM t3
    ),

    t4 AS (SELECT 
    TO_CHAR(data_consulta, 'YYYY-MM') AS ano_mes,
    especialidade,
    count(status) AS qtd_status
    FROM case_2.consultas_silver
    GROUP BY ano_mes, especialidade),

    t5 AS (SELECT
    meses.ano_mes,
    meses.especialidade,
    COALESCE(t4.qtd_status,0) AS qtd_status
    FROM meses
    LEFT JOIN t4
    ON t4.ano_mes=meses.ano_mes
    AND t4.especialidade=meses.especialidade
    ORDER BY ano_mes),

    t5_fix AS (SELECT *,
    SUM(COALESCE(qtd_status,0)) OVER (PARTITION BY especialidade ORDER BY ano_mes) AS qtd_status_acum
    FROM t5),

    t6 AS (SELECT 
    t3_fix.ano_mes,
    t3_fix.especialidade,
    COALESCE(1. * t3_fix.qtd_cancelado/t5_fix.qtd_status,0) AS tx_cancelado,
    COALESCE(1. * t3_fix.qtd_cancelado_acum/t5_fix.qtd_status_acum,0) AS tx_cancelado_acum
    FROM t3_fix
    LEFT JOIN t5_fix
    ON t3_fix.ano_mes=t5_fix.ano_mes
    AND t3_fix.especialidade=t5_fix.especialidade),

    t7 AS (
        SELECT *,
        LAG(tx_cancelado) OVER (PARTITION BY especialidade ORDER BY ano_mes) AS tx_mes_anterior,
        tx_cancelado - LAG(tx_cancelado) OVER (PARTITION BY especialidade ORDER BY ano_mes) AS variacao
        FROM t6
    )

    SELECT *
    FROM t7



"""
Q4 = pd.read_sql(query_4, engine)

Q4

,ano_mes,especialidade,tx_cancelado,tx_cancelado_acum,tx_mes_anterior,variacao
0,2022-01,cardiologia,0.000000,0.000000,NaN,NaN
1,2022-02,cardiologia,0.500000,0.142857,0.000000,0.500000
2,2022-03,cardiologia,0.333333,0.200000,0.500000,-0.166667
3,2022-04,cardiologia,0.666667,0.307692,0.333333,0.333333
4,2022-05,cardiologia,0.333333,0.312500,0.666667,-0.333333
...,...,...,...,...,...,...
149,2023-05,psiquiatria,0.000000,0.142857,0.500000,-0.500000
150,2023-06,psiquiatria,0.000000,0.136364,0.000000,0.000000
151,2023-07,psiquiatria,0.000000,0.130435,0.000000,0.000000
152,2023-08,psiquiatria,0.000000,0.125000,0.000000,0.000000
